### DAM

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
import dam_mo
import numpy as np

max_episode_steps = 100

def make_env():
    env = gym.make('Dam_MO', nO=[0]) # [1]
    env = TimeLimit(env, max_episode_steps)
    return env

In [ ]:
from stable_baselines3 import PPO
env = make_env()

model = PPO("MlpPolicy", env, learning_rate=1e-2, ent_coef=0.1, verbose=1)
model.learn(total_timesteps=10_000_000)

In [ ]:
# Evaluate the trained agent
obs, info = env.reset()
total_reward = 0
for _ in range(max_episode_steps):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action: {action}, Reward: {reward}, Info: {info}")
    total_reward += float(reward)
    if terminated or truncated:
        break
print(f"Total Reward: {total_reward}")

In [ ]:
model.save("ppo_dam2_reward1")

### Ensemble

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
import dam_mo
import numpy as np

max_episode_steps = 100

def make_env():
    env = gym.make('Dam_MO', nO=[0,1])
    env = TimeLimit(env, max_episode_steps)
    return env
env = make_env()

In [ ]:
from ensemble import *

trained_ensemble = train_with_ppo("ppo_dam2_reward0", "ppo_dam2_reward1", env, 1000000)

In [ ]:
trained_ensemble.save("ensemble_ppo_model")

### WATER RESERVOIR (test)

In [ ]:
import mo_gymnasium as mo_gym
from stable_baselines3 import PPO
from gymnasium.wrappers import TimeLimit
import water_reservoir

In [ ]:
env = mo_gym.make("water-reservoir-meta", render_mode="human", nO=0, normalized_action=True)
env = TimeLimit(env, max_episode_steps=10)

In [ ]:
model = PPO("MlpPolicy", env, learning_rate=1e-2, ent_coef=0.1, verbose=2)
model.learn(total_timesteps=10_000_000)